# Epistemic Crucible — Diagnostic Experiment

This notebook runs four baseline agents on the **affordance** task family,
computes the full diagnostic metric vector, and visualises three key signals:

1. **Shortcut Exposure** — train vs test Task Success Rate per agent  
2. **Intervention Trace** — which steps produced causal effects  
3. **Failure Mode Breakdown** — how each agent fails

The experiment is fully deterministic for fixed seeds and makes no network requests.

**Prerequisites:**
```bash
pip install -e .[notebooks]   # adds matplotlib, pandas
```

## 1. Setup

In [ ]:
import json
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

# Locate repo root regardless of notebook CWD.
_REPO_ROOT = pathlib.Path().resolve()
while not (_REPO_ROOT / "crucible").is_dir() and _REPO_ROOT.parent != _REPO_ROOT:
    _REPO_ROOT = _REPO_ROOT.parent
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

_RESULTS = _REPO_ROOT / "results"
_RESULTS.mkdir(exist_ok=True)
print(f"Repo root : {_REPO_ROOT}")
print(f"Results   : {_RESULTS}")

## 2. Run the diagnostic experiment

Agents: `random`, `heuristic`, `memorization`, `hybrid_rule_planner`  
Family: `affordance` (seeds 0–9, 8 train / 2 test per the 80/20 split)

Seeds control both world generation and agent randomness, so results are
identical across runs.

In [ ]:
from experiments.run_diagnostic import main as run_diagnostic

_AGENTS = ["random", "heuristic", "memorization", "hybrid_rule_planner"]
_SEEDS  = list(range(10))

result = run_diagnostic([
    "--families", "affordance",
    "--seeds",    *[str(s) for s in _SEEDS],
    "--agents",   *_AGENTS,
    "--output-dir", str(_RESULTS),
])

_TRACE_PATH  = result["trace"]
_REPORT_PATH = result["report"]
print(f"Trace  : {_TRACE_PATH}")
print(f"Report : {_REPORT_PATH}")

In [ ]:
from crucible.metrics import filter_records, load_traces, task_success_rate

_steps, _outcomes = load_traces([_TRACE_PATH])
print(f"Loaded {len(_steps)} step records, {len(_outcomes)} outcome records")

## 3. Plot 1 — Shortcut Exposure

Train and test Task Success Rate per agent on the affordance family.
A gap (train > test) reveals shortcut exploitation — the agent learned
the train-specific colour cue rather than the hidden conductivity property.

In [ ]:
train_tsrs = [
    task_success_rate(_outcomes, family="affordance", agent=a, split="train").value
    for a in _AGENTS
]
test_tsrs = [
    task_success_rate(_outcomes, family="affordance", agent=a, split="test").value
    for a in _AGENTS
]

x     = np.arange(len(_AGENTS))
width = 0.35
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - width / 2, train_tsrs, width, label="Train", color="#4472C4")
ax.bar(x + width / 2, test_tsrs,  width, label="Test",  color="#ED7D31")
ax.set_xticks(x)
ax.set_xticklabels([a.replace("_", "\n") for a in _AGENTS])
ax.set_ylabel("Task Success Rate")
ax.set_ylim(0, 1.15)
ax.set_title("Shortcut Exposure — Affordance Family\nTrain vs Test TSR")
ax.legend()
fig.tight_layout()
_out = _RESULTS / "plot_shortcut_exposure.png"
fig.savefig(str(_out), dpi=100)
plt.show()
print(f"Saved → {_out}")

# Summarise the gap.
for agent, tr, te in zip(_AGENTS, train_tsrs, test_tsrs):
    gap = tr - te
    flag = " ← shortcut" if gap > 0.1 else ""
    print(f"  {agent:<25}  train={tr:.2f}  test={te:.2f}  SS={gap:+.2f}{flag}")

## 4. Plot 2 — Intervention Trace Heatmap

Each row is one episode; each column is a step.  
**Green** = intervention (apply/combine/inspect) that produced a causal effect.  
**Amber** = intervention with no effect.  
**White** = non-intervention step.

Agents with genuine causal understanding produce more green; random agents
show scattered amber with occasional lucky green squares.

In [ ]:
_INTERVENTION_KINDS = {"apply", "combine", "inspect"}
_MAX_STEP = 30
_N_EP     = 5  # episodes to show per agent

fig, axes = plt.subplots(len(_AGENTS), 1, figsize=(14, 2.5 * len(_AGENTS)))

for i, agent_name in enumerate(_AGENTS):
    agent_steps = filter_records(_steps, agent=agent_name, family="affordance")

    # Group steps by episode.
    ep_map: dict[int, list[dict]] = {}
    for s in agent_steps:
        ep_map.setdefault(s["episode"], []).append(s)

    ep_ids = sorted(ep_map)[:_N_EP]
    matrix = np.zeros((len(ep_ids), _MAX_STEP))

    for row, ep_id in enumerate(ep_ids):
        for s in ep_map[ep_id]:
            col = s["step"]
            if col >= _MAX_STEP:
                continue
            kind = s["action"]["kind"]
            if kind in _INTERVENTION_KINDS:
                matrix[row, col] = 2 if s["effects"] else 1

    ax = axes[i]
    cmap = plt.cm.get_cmap("RdYlGn", 3)
    ax.imshow(matrix, aspect="auto", cmap=cmap, vmin=0, vmax=2,
              interpolation="nearest")
    ax.set_title(f"{agent_name}", fontsize=10, loc="left")
    ax.set_xlabel("Step")
    ax.set_ylabel("Episode")
    ax.set_yticks(range(len(ep_ids)))

fig.suptitle(
    "Intervention Trace Heatmap (green=effect produced, amber=no effect, white=no intervention)",
    y=1.01, fontsize=11
)
fig.tight_layout()
_out = _RESULTS / "plot_intervention_trace.png"
fig.savefig(str(_out), dpi=100, bbox_inches="tight")
plt.show()
print(f"Saved → {_out}")

## 5. Plot 3 — Failure Mode Breakdown

Stacked horizontal bars showing how many failed episodes fall into each
failure mode per agent. A single dominant mode indicates stereotyped
failure; diverse modes suggest unfocused exploration.

In [ ]:
from crucible.metrics import failure_diversity

_MODES  = ["timeout", "no_interaction", "high_illegal", "no_effects", "energy_depleted"]
_COLORS = ["#4472C4", "#ED7D31", "#A9D18E", "#FF4444", "#7030A0"]

fig, ax = plt.subplots(figsize=(9, 5))

for i, agent_name in enumerate(_AGENTS):
    a_steps    = filter_records(_steps,    agent=agent_name, family="affordance")
    a_outcomes = filter_records(_outcomes, agent=agent_name, family="affordance")
    fd = failure_diversity(a_steps, a_outcomes)

    left = 0
    for mode, color in zip(_MODES, _COLORS):
        count = fd.value.get(mode, 0)
        label = mode if i == 0 else None
        ax.barh(i, count, left=left, color=color, label=label, edgecolor="white")
        if count > 0:
            ax.text(left + count / 2, i, str(count), ha="center", va="center",
                    fontsize=8, color="white", fontweight="bold")
        left += count

ax.set_yticks(range(len(_AGENTS)))
ax.set_yticklabels(_AGENTS)
ax.set_xlabel("Failed episodes")
ax.set_title("Failure Mode Breakdown — Affordance Family")
ax.legend(loc="lower right", fontsize=8)
fig.tight_layout()
_out = _RESULTS / "plot_failure_modes.png"
fig.savefig(str(_out), dpi=100)
plt.show()
print(f"Saved → {_out}")

## 6. Full Metric Report

The JSON report produced by `run_diagnostic` contains all eight metrics.
No single aggregate score is reported — the vector is the evaluation.

In [ ]:
import pandas as pd

report = json.loads(_REPORT_PATH.read_text())

rows = []
for name, r in report.items():
    val = r["value"]
    if isinstance(val, float):
        display = f"{val:.4f}"
    elif isinstance(val, dict):
        items = list(val.items())[:2]
        display = ", ".join(f"{k}: {v}" for k, v in items)
        if len(val) > 2:
            display += " …"
    else:
        display = str(val)
    rows.append({"metric": name, "value": display, "n": r["count"]})

df = pd.DataFrame(rows).set_index("metric")
print(df.to_string())

## Key findings

- **Shortcut Exposure**: agents with `train TSR > test TSR` on the affordance
  family are exploiting the colour-conductivity correlation present in training
  worlds. The gap (Shortcut Sensitivity) measures this directly.

- **Intervention Trace**: the heatmap exposes whether successful episodes come
  from systematic causal reasoning (dense green) or lucky guessing (sparse
  green amid amber).

- **Failure Diversity**: a single dominant failure mode indicates stereotyped
  behaviour; diverse modes suggest unfocused exploration without convergence.

Task success rate alone cannot distinguish these failure patterns.
The full metric vector is required.